In [0]:
%run ./utils

# Import

In [0]:
from datetime import date
import holidays

# Load

In [0]:
dbutils.widgets.text("silver_table_name", "tb_chamados_hora")
silver_table_name = dbutils.widgets.get("silver_table_name")

COLUNAS_HORA = ["hora_abertura_chamado", "hora_inicio_atendimento", "hora_finalizacao_atendimento"]

# Só as colunas de hora: a dimensão não usa mais nada da Silver
df_src = load_df(silver_table_name).select(*COLUNAS_HORA)

# Tratamento

## Intervalo do calendário

In [0]:
# Criar calendario do tempo de referencia
bounds = df_src.agg(
    F.min(F.least("hora_abertura_chamado", "hora_inicio_atendimento","hora_finalizacao_atendimento")).alias("dt_min"),
    F.max(F.greatest("hora_abertura_chamado", "hora_inicio_atendimento","hora_finalizacao_atendimento")).alias("dt_max"),
).first()

if bounds.dt_min is None:
    raise ValueError(f"{silver_table_name} não tem nenhum timestamp válido")

ano_ini,ano_fim = bounds.dt_min.year,bounds.dt_max.year
print(f"Calendário: {ano_ini} a {ano_fim}")

## Calendário e feriados

In [0]:
df_cal = spark.range(1).select(
    F.explode(F.sequence(
        F.lit(date(ano_ini, 1, 1)), F.lit(date(ano_fim, 12, 31)), F.expr("INTERVAL 1 DAY")
    )).alias("data")
)

In [0]:
# Feriados nacionais (sem subdiv, a lib retorna só os do país)
anos = range(ano_ini, ano_fim + 1)
br = holidays.Brazil(years=anos, language="pt_BR")

# Sanidade: todo ano precisa ter Natal e Tiradentes
for ano in anos:
    assert date(ano, 12, 25) in br and date(ano, 4, 21) in br, f"Feriados nacionais incompletos em {ano}"

# A chave do dict é a data, então o join com o calendário continua 1:1
df_fer = spark.createDataFrame(sorted(br.items()), "data date, nome_feriado string")

## Atributos da data

In [0]:
nomes_dia = ["Domingo", "Segunda-feira", "Terça-feira", "Quarta-feira", "Quinta-feira", "Sexta-feira", "Sábado"]
nomes_mes = ["Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
             "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"]

def nome_por_indice(nomes, indice):
    # indice é 1-based, como dayofweek e month
    return F.element_at(F.array(*[F.lit(n) for n in nomes]), indice)

df_dim = (
    df_cal
    .join(df_fer, "data", "left")      
    .withColumn("data_sk", F.date_format("data", "yyyyMMdd").cast("bigint"))
    .withColumn("ano", F.year("data"))
    .withColumn("trimestre", F.quarter("data"))
    .withColumn("mes", F.month("data"))
    .withColumn("nome_mes", nome_por_indice(nomes_mes, F.col("mes")))
    .withColumn("ano_mes", F.date_format("data", "yyyyMM").cast("int"))
    .withColumn("dia", F.dayofmonth("data"))
    .withColumn("dia_semana_num", F.dayofweek("data"))      # 1 = domingo ... 7 = sábado
    .withColumn("dia_semana", nome_por_indice(nomes_dia, F.col("dia_semana_num")))
    .withColumn("fim_de_semana", F.col("dia_semana_num").isin(1, 7))
    .withColumn("feriado", F.col("nome_feriado").isNotNull())
    .withColumn("dia_util", ~F.col("fim_de_semana") & ~F.col("feriado"))
    .select("data_sk", "data", "ano", "trimestre", "mes", "nome_mes", "ano_mes", "dia",
            "dia_semana_num", "dia_semana", "fim_de_semana", "feriado", "nome_feriado", "dia_util")
)

# Surrogate Keys

In [0]:
df_unknown = spark.range(1).select(
    F.lit(-1).cast("bigint").alias("data_sk"),
    F.lit(None).cast("date").alias("data"),
    F.lit(None).cast("int").alias("ano"),
    F.lit(None).cast("int").alias("trimestre"),
    F.lit(None).cast("int").alias("mes"),
    F.lit("NÃO INFORMADO").alias("nome_mes"),
    F.lit(None).cast("int").alias("ano_mes"),
    F.lit(None).cast("int").alias("dia"),
    F.lit(None).cast("int").alias("dia_semana_num"),
    F.lit("NÃO INFORMADO").alias("dia_semana"),
    F.lit(False).alias("fim_de_semana"),
    F.lit(False).alias("feriado"),
    F.lit(None).cast("string").alias("nome_feriado"),
    F.lit(False).alias("dia_util"),
)

df_dim = df_dim.unionByName(df_unknown)

# Verificações

In [0]:
# Uma única passada; os asserts falham de verdade e interrompem a task do job
n_dias = (date(ano_fim, 12, 31) - date(ano_ini, 1, 1)).days + 1

stats = df_dim.agg(
    F.count("*").alias("linhas"),
    F.countDistinct("data_sk").alias("sks"),
    F.countDistinct("data").alias("datas"),
    F.sum(F.col("data_sk").isNull().cast("int")).alias("sk_nula"),
    F.sum((F.col("data_sk") == -1).cast("int")).alias("desconhecidos"),
    F.sum(F.col("feriado").cast("int")).alias("feriados"),
).first()

print(f"Linhas: {stats.linhas} (esperado {n_dias + 1}) | SKs únicas: {stats.sks} | "
      f"datas únicas: {stats.datas} | feriados: {stats.feriados}")

assert stats.linhas == n_dias + 1, f"Esperado {n_dias + 1} linhas (dias + membro -1), obtido {stats.linhas}"
assert stats.sks == stats.linhas,  "data_sk duplicada"
assert stats.datas == n_dias,      "data duplicada ou faltando"
assert stats.sk_nula == 0,         "data_sk nula"
assert stats.desconhecidos == 1,   "membro desconhecido (-1) ausente ou duplicado"

display(df_dim.orderBy("data_sk").limit(10))

# Salvamento

In [0]:
dbutils.widgets.text("gold_table_name", "dim_data")
gold_table_name = dbutils.widgets.get("gold_table_name")

In [0]:
save_df(df_dim, gold_table_name)

spark.sql(f"COMMENT ON TABLE {gold_schema}.{gold_table_name} IS "
          "'Dimensão de data. Grain: 1 por dia. SK yyyyMMdd; -1 = não informado. Feriados nacionais apenas.'")

In [0]:
display(spark.table(f"{gold_schema}.{gold_table_name}").orderBy("data_sk").limit(10))   